In [ ]:
#| default_exp handlers.pipeline.intake

# Intake

Boundary ingestion planning and execution for default CSV/TSV sources, with an explicit Early Excel Intake Gate and custom-loader delegation.

In [ ]:
#| export
from __future__ import annotations
import io
from pathlib import Path
from typing import Optional
import pandas as pd
import requests
from pydantic import BaseModel
from marisco.handlers.pipeline.contracts import HandlerConfig, PluginSpec
from marisco.handlers.pipeline.gates import _custom_loader_skeleton

## Intake Plan

In [ ]:
#| export
class IntakePlan(BaseModel):
    "Describe the boundary ingestion route chosen for a handler config."
    kind: str
    fmt: str
    grp: str = "SEAWATER"
    sep: str = ","


def _unsupported_default_loader_message(cfg: HandlerConfig) -> str:
    fmt = (cfg.fmt or "csv").lower()
    sections = [
        f"Intake Gate failed: format '{fmt}' cannot be loaded by the default CSV reader.",
        "The default ingestion path only supports text-delimited CSV/TSV files.",
        "Excel workbooks require a Custom Boundary Loader before the declarative pipeline can begin.",
        _custom_loader_skeleton(cfg, findings=[{'grp': 'SEAWATER'}]),
    ]
    return "\n\n".join(part for part in sections if part)


def resolve_intake_plan(cfg: HandlerConfig, grp: str = "SEAWATER") -> IntakePlan:
    fmt = (cfg.fmt or "csv").lower()
    if fmt in {"xlsx", "xls", "excel"}:
        return IntakePlan(kind="unsupported_excel", fmt=fmt, grp=grp)
    sep = "\t" if fmt == "tsv" else ","
    return IntakePlan(kind="delimited", fmt=fmt, grp=grp, sep=sep)


def execute_intake_plan(cfg: HandlerConfig, plan: IntakePlan) -> dict[str, pd.DataFrame]:
    if plan.kind == "unsupported_excel":
        msg = _unsupported_default_loader_message(cfg)
        print(f"\n{msg}\n")
        raise ValueError(msg)
    r = requests.get(cfg.url, timeout=60)
    r.raise_for_status()
    return {plan.grp: pd.read_csv(io.BytesIO(r.content), sep=plan.sep)}


def load_data(cfg: HandlerConfig, grp: str = "SEAWATER") -> dict[str, pd.DataFrame]:
    "Fetch raw provider data through the default boundary-ingestion route."
    plan = resolve_intake_plan(cfg, grp=grp)
    return execute_intake_plan(cfg, plan)


def resolve_loader_fn(spec: PluginSpec, yaml_dir: Path = None):
    "Resolve a custom boundary-loader function from a PluginSpec."
    return spec.resolve_fn(yaml_dir)


def call_loader(cfg: HandlerConfig, yaml_dir: Optional[Path] = None):
    "Resolve the configured boundary loader and execute it with cfg plus loader args."
    if not cfg.loader:
        return load_data(cfg)
    loader_fn = resolve_loader_fn(cfg.loader, yaml_dir=yaml_dir)
    return loader_fn(cfg, **cfg.loader.args)

In [ ]:
plan = resolve_intake_plan(HandlerConfig.from_yaml("config/handlers/fram_strait.yaml"))
print(plan.kind)
print(plan.fmt)